## **Proceso Extracción - Transformacion**

### **1. Importación de librerías**

In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import time

### **2. Configuración de la conexión**

In [2]:
USUARIO = "postgres"
PASSWORD = "1028"
HOST = "localhost"
PUERTO = "5432"
BASE_DATOS = "dw_analisis_criminalidad"

engine = create_engine(
    f"postgresql+psycopg2://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{BASE_DATOS}"
)

print("Conexión creada correctamente.")

Conexión creada correctamente.


Prueba de conexión

In [3]:
query = "SELECT * FROM dim_delito"

df = pd.read_sql(query, engine)
df.head()

,id_delito,tipo_delito
0,1,Amenazas
1,2,Delitos sexuales
2,3,Homicidio
3,4,Hurto a residencias y entidades comerciales
4,5,Hurto de motocicletas y automotores


### **3. Extracción de datos**

Nombre para columna tipo delito de acuerdo con nombre de archivo

In [4]:
TIPOS_DELITO = {
    "Reporte amenazas.csv": "Amenazas",
    "Reporte delitos sexuales.csv": "Delitos sexuales",
    "Reporte homicidios.csv": "Homicidio",
    "Reporte hurto motocicletas y automotores.csv": "Hurto de motocicletas y automotores",
    "Reporte hurto residencias y entidades comerciales.csv": "Hurto a residencias y entidades comerciales",
    "Reporte lesiones personales y accidente de transito.csv": "Lesiones personales",
    "Reporte violencia intrafamiliar.csv": "Violencia intrafamiliar"
}

Leer y transformar archivos

In [5]:
# Ruta de los archivos CSV
RUTA_DATOS_CSV = Path("Datos CSV")

In [6]:
archivos = sorted(RUTA_DATOS_CSV.glob("*.csv"))

print(f"Se encontraron {len(archivos)} archivos:\n")

for archivo in archivos:
    print(f"- {archivo.name}")

Se encontraron 7 archivos:

- Reporte amenazas.csv
- Reporte delitos sexuales.csv
- Reporte homicidios.csv
- Reporte hurto motocicletas y automotores.csv
- Reporte hurto residencias y entidades comerciales.csv
- Reporte lesiones personales y accidente de transito.csv
- Reporte violencia intrafamiliar.csv


In [7]:
dataframes = []

for archivo in archivos:

    print(f"Leyendo {archivo.name}...")

    df = pd.read_csv(
        archivo,
        encoding="utf-8-sig",
        low_memory=False
    )

    # Estandarizar nombres de columnas
    df.columns = (
        df.columns
          .str.strip()
          .str.upper()
    )

    # Corregir diferencias del archivo de homicidios
    if "GRUPO EDAD" in df.columns:

        df.rename(
            columns={
                "GRUPO EDAD": "GRUPO ETARIO"
            },
            inplace=True
        )

    # Eliminar columnas que no se utilizarán
    df.drop(
        columns=[
            "CODIGO DANE",
            "DELITOS",
            "DELITO",
            "TIPO DE HURTO",
            "DESCRIPCIÓN CONDUCTA"
        ],
        errors="ignore",
        inplace=True
    )

    # Crear nuestra columna DELITO
    df["DELITO"] = TIPOS_DELITO[archivo.name]

    # Homologar Bogotá D.C. en el archivo de homicidios
    if archivo.name == "Reporte homicidios.csv":
        mask_bogota = (
            df["MUNICIPIO"]
            .astype("string")
            .str.strip()
            .str.upper()
            .eq("BOGOTÁ D.C. (CT)")
        )

        df.loc[mask_bogota, "DEPARTAMENTO"] = "Bogotá D.C."
        df.loc[mask_bogota, "MUNICIPIO"] = "Bogotá"

    dataframes.append(df)

    print(f"   Registros: {len(df):,}")

Leyendo Reporte amenazas.csv...
   Registros: 650,347
Leyendo Reporte delitos sexuales.csv...
   Registros: 392,576
Leyendo Reporte homicidios.csv...
   Registros: 154,428
Leyendo Reporte hurto motocicletas y automotores.csv...
   Registros: 641,663
Leyendo Reporte hurto residencias y entidades comerciales.csv...
   Registros: 633,804
Leyendo Reporte lesiones personales y accidente de transito.csv...
   Registros: 1,351,421
Leyendo Reporte violencia intrafamiliar.csv...
   Registros: 682,558


Unificar todos los dataframes

In [8]:
df = pd.concat(
    dataframes,
    ignore_index=True
)

Verificar resultado

In [9]:
print(f"\nTotal de registros: {len(df):,}")

print("\nColumnas:")

print(df.columns.tolist())


Total de registros: 4,506,797

Columnas:
['DEPARTAMENTO', 'MUNICIPIO', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD', 'DELITO']


Visualizar primeros registros

In [10]:
df.head()

,DEPARTAMENTO,MUNICIPIO,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD,DELITO
0,SUCRE,Coveñas,NO REPORTADO,22/06/2013,MASCULINO,ADULTOS,1,Amenazas
1,ATLÁNTICO,Barranquilla (CT),NO REPORTADO,13/12/2013,MASCULINO,ADULTOS,1,Amenazas
2,CAQUETÁ,Florencia (CT),NO REPORTADO,12/06/2012,FEMENINO,ADULTOS,1,Amenazas
3,VALLE,Cartago,NO REPORTADO,01/04/2012,MASCULINO,ADULTOS,1,Amenazas
4,VALLE,Cali (CT),NO REPORTADO,20/09/2012,FEMENINO,ADULTOS,1,Amenazas


### **4. Transformación**

Se define el periodo de estudio 

In [11]:
ANIO_INICIO = 2014
ANIO_FIN = 2024

Informacion general del dataframe

In [12]:
print(f"Registros: {len(df):,}")
print(f"Columnas: {len(df.columns)}")

Registros: 4,506,797
Columnas: 8


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4506797 entries, 0 to 4506796
Data columns (total 8 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   DEPARTAMENTO  object
 1   MUNICIPIO     object
 2   ARMAS MEDIOS  object
 3   FECHA HECHO   object
 4   GENERO        object
 5   GRUPO ETARIO  object
 6   CANTIDAD      int64 
 7   DELITO        object
dtypes: int64(1), object(7)
memory usage: 275.1+ MB


In [14]:
df.describe(include="all")

,DEPARTAMENTO,MUNICIPIO,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD,DELITO
count,4506797,4506769,4506797,4506797,3926133,4447274,4.506797e+06,4506797
unique,34,1024,53,5934,3,3,NaN,7
top,BOGOTA,Bogotá D.C. (CT),SIN EMPLEO DE ARMAS,01/01/2017,MASCULINO,ADULTOS,NaN,Lesiones personales
freq,567222,567222,1775380,1991,2116257,4080236,NaN,1351421
mean,NaN,NaN,NaN,NaN,NaN,NaN,1.377545e+00,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,2.420378e+00,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN


#### **4.1. Renombrar columnas**

In [15]:
df.columns = [
    "departamento",
    "municipio",
    "armas_medios",
    "fecha_hecho",
    "genero",
    "grupo_etario",
    "cantidad",
    "delito"
]

#### **4.2. Convertir tipos de datos**

In [16]:
df["fecha_hecho"] = pd.to_datetime(
    df["fecha_hecho"],
    errors="coerce"
)

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_17488\933075919.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["fecha_hecho"] = pd.to_datetime(


In [17]:
df["cantidad"] = pd.to_numeric(
    df["cantidad"],
    errors="coerce"
)

In [18]:
df.dtypes

departamento            object
municipio               object
armas_medios            object
fecha_hecho     datetime64[ns]
genero                  object
grupo_etario            object
cantidad                 int64
delito                  object
dtype: object

In [19]:
columnas_texto = [
    "departamento",
    "municipio",
    "armas_medios",
    "genero",
    "grupo_etario",
    "delito"
]

for columna in columnas_texto:

    df[columna] = (
        df[columna]
        .astype("string")
        .str.strip()
    )

In [20]:
df.head()

,departamento,municipio,armas_medios,fecha_hecho,genero,grupo_etario,cantidad,delito
0,SUCRE,Coveñas,NO REPORTADO,2013-06-22,MASCULINO,ADULTOS,1,Amenazas
1,ATLÁNTICO,Barranquilla (CT),NO REPORTADO,2013-12-13,MASCULINO,ADULTOS,1,Amenazas
2,CAQUETÁ,Florencia (CT),NO REPORTADO,2012-06-12,FEMENINO,ADULTOS,1,Amenazas
3,VALLE,Cartago,NO REPORTADO,2012-04-01,MASCULINO,ADULTOS,1,Amenazas
4,VALLE,Cali (CT),NO REPORTADO,2012-09-20,FEMENINO,ADULTOS,1,Amenazas


#### **4.3. Revision de columnas y manejo de nulos**

##### **Genero**

In [21]:
df["genero"].value_counts(dropna=False)

genero
MASCULINO     2116257
FEMENINO      1807477
<NA>           580664
NO REPORTA       2399
Name: count, dtype: Int64

En este caso, un NA se puede relacionar con un no reportado

In [22]:
# Reemplazar valores faltantes y "NO REPORTA"

df["genero"] = (
    df["genero"]
        .fillna("NO REPORTADO")
        .replace({
            "NO REPORTA": "NO REPORTADO"
        })
)

In [23]:
df["genero"] = df["genero"].replace({
    "MASCULINO": "Masculino",
    "FEMENINO": "Femenino",
    "NO REPORTADO": "No reportado"
})

Se estandariza el texto con mayuscula inicial

In [24]:
df["genero"].value_counts(dropna=False)

genero
Masculino       2116257
Femenino        1807477
No reportado     583063
Name: count, dtype: Int64

##### **Grupo etario**

In [25]:
df["grupo_etario"].value_counts(dropna=False)

grupo_etario
ADULTOS         4080236
ADOLESCENTES     224688
MENORES          142350
<NA>              59523
Name: count, dtype: Int64

Los NA se tratan como no reportado y se modifica a mayuscula inicial

In [26]:
# Estandarizar grupo etario
df["grupo_etario"] = (
    df["grupo_etario"]
        .fillna("NO REPORTADO")
        .replace({
            "ADULTOS": "Adultos",
            "ADOLESCENTES": "Adolescentes",
            "MENORES": "Menores",
            "NO REPORTADO": "No reportado"
        })
)

In [27]:
df["grupo_etario"].value_counts(dropna=False)

grupo_etario
Adultos         4080236
Adolescentes     224688
Menores          142350
No reportado      59523
Name: count, dtype: Int64

##### **Armas**

In [28]:
df["armas_medios"].value_counts(dropna=False)

armas_medios
SIN EMPLEO DE ARMAS                   1775380
CONTUNDENTES                           919974
ARMA DE FUEGO                          480933
ARMA BLANCA / CORTOPUNZANTE            408852
NO REPORTADO                           287172
LLAVE MAESTRA                          228429
VEHICULO                               185133
MOTO                                   123168
PALANCAS                                39441
ESCOPOLAMINA                            16357
ACIDO                                   13286
PUNZANTES                                5754
CORTANTES                                3875
BICICLETA                                2812
ARMA TRAUMATICA                          2300
PERRO                                    2298
ARTEFACTO EXPLOSIVO/CARGA DINAMITA       1766
CORTOPUNZANTES                           1465
LICOR ADULTERADO                         1012
COMBUSTIBLE                               940
ALIMENTOS VENCIDOS                        745
GRANADA DE MANO      

In [29]:
# Estandarizar categorías de armas o medios

df["armas_medios"] = (
    df["armas_medios"]
        .fillna("No reportado")
        .replace({
            "NO REPORTADO": "No reportado",

            "ARMAS BLANCAS": "ARMA BLANCA / CORTOPUNZANTE",
            "CORTOPUNZANTES": "ARMA BLANCA / CORTOPUNZANTE",
            "CUCHILLA": "ARMA BLANCA / CORTOPUNZANTE",

            "POLVORA(FUEGOS PIROTECNICOS)": "POLVORA (FUEGOS PIROTECNICOS)"
        })
)

##### **Departamento**

In [30]:
df["departamento"].nunique()

34

In [31]:
df["departamento"].sort_values().unique()

<StringArray>
[          'AMAZONAS',          'ANTIOQUIA',             'ARAUCA',
          'ATLÁNTICO',             'BOGOTA',            'BOLÍVAR',
             'BOYACÁ',        'Bogotá D.C.',             'CALDAS',
            'CAQUETÁ',           'CASANARE',              'CAUCA',
              'CESAR',              'CHOCÓ',       'CUNDINAMARCA',
            'CÓRDOBA',            'GUAINÍA',            'GUAJIRA',
           'GUAVIARE',              'HUILA',          'MAGDALENA',
               'META',             'NARIÑO', 'NORTE DE SANTANDER',
           'PUTUMAYO',            'QUINDÍO',          'RISARALDA',
         'SAN ANDRÉS',          'SANTANDER',              'SUCRE',
             'TOLIMA',              'VALLE',             'VAUPÉS',
            'VICHADA']
Length: 34, dtype: string

Se escribe como titulo y se modifican nombres oficiales

In [32]:
# Cambiar a formato de presentación
df["departamento"] = df["departamento"].str.title()

# Correcciones de nombres oficiales
df["departamento"] = df["departamento"].replace({
    "Bogota": "Bogotá D.C.",
    "Guajira": "La Guajira",
    "Valle": "Valle del Cauca"
})

##### **Municipio**

In [33]:
print(f"Municipios únicos: {df['municipio'].nunique():,}")
print(f"Registros nulos: {df['municipio'].isna().sum():,}")

Municipios únicos: 1,024
Registros nulos: 28


Se cambian nulos por "No reportado"

In [34]:
df["municipio"] = df["municipio"].fillna("No reportado")

In [35]:
df["municipio"] = (
    df["municipio"].str.title()
)

In [36]:
df["municipio"] = (
    df["municipio"]
        .str.replace(" Del ", " del ", regex=False)
        .str.replace(" De ", " de ", regex=False)
        .str.replace(" La ", " la ", regex=False)
        .str.replace(" Las ", " las ", regex=False)
        .str.replace(" Los ", " los ", regex=False)
)

In [37]:
municipios = sorted(df["municipio"].dropna().unique())

print(f"Total de municipios: {len(municipios)}")
pd.Series(municipios).to_csv(
    "Salidas/municipios_unicos.csv",
    index=False,
    header=["municipio"],
    encoding="utf-8-sig"
)

Total de municipios: 1025


##### **Fecha**

In [38]:
print(df["fecha_hecho"].isna().sum())

print(df["fecha_hecho"].min())
print(df["fecha_hecho"].max())

0
2010-01-01 00:00:00
2026-03-31 00:00:00


Delimitacion por periodo de estudio

In [39]:
df.shape

(4506797, 8)

In [40]:
# Filtrar únicamente el periodo de estudio
df = df[
    (df["fecha_hecho"].dt.year >= ANIO_INICIO) &
    (df["fecha_hecho"].dt.year <= ANIO_FIN)
].copy()

In [41]:
print(df["fecha_hecho"].min())
print(df["fecha_hecho"].max())

2014-01-01 00:00:00
2024-12-31 00:00:00


In [42]:
print(f"Registros para el estudio: {len(df):,}")

Registros para el estudio: 3,481,420


##### **Validacion de nulos**

In [43]:
# Conteo de valores nulos
nulos = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .rename("Valores nulos")
)

print(nulos)

departamento    0
municipio       0
armas_medios    0
fecha_hecho     0
genero          0
grupo_etario    0
cantidad        0
delito          0
Name: Valores nulos, dtype: int64


#### **4.4. Consolidación de registros**

Con el fin de garantizar una granularidad consistente en la tabla de hechos del Data Warehouse, los registros se consolidan agrupándolos por todas las dimensiones definidas (tiempo, ubicación, delito, arma o medio y características de la víctima). Para cada combinación única de dimensiones, la medida **cantidad** se obtiene mediante la suma de los valores correspondientes. Esta transformación reduce la redundancia en los datos sin alterar el número total de delitos reportados por la fuente.

In [44]:
# Definir las dimensiones de la tabla de hechos
columnas_dimension = [
    "fecha_hecho",
    "departamento",
    "municipio",
    "delito",
    "armas_medios",
    "genero",
    "grupo_etario"
]

# Consolidar registros con la misma combinación de dimensiones
df = (
    df.groupby(columnas_dimension, as_index=False)
      .agg(cantidad=("cantidad", "sum"))
)

# Verificar el resultado
print(f"Registros después de la consolidación: {len(df):,}")
print(f"Total de delitos: {df['cantidad'].sum():,}")

df.head()

Registros después de la consolidación: 2,786,613
Total de delitos: 4,805,136


,fecha_hecho,departamento,municipio,delito,armas_medios,genero,grupo_etario,cantidad
0,2014-01-01,Antioquia,Amagá,Amenazas,SIN EMPLEO DE ARMAS,Femenino,Adultos,1
1,2014-01-01,Antioquia,Amagá,Lesiones personales,ARMA BLANCA / CORTOPUNZANTE,Masculino,Adultos,1
2,2014-01-01,Antioquia,Andes,Lesiones personales,CONTUNDENTES,Femenino,Adultos,3
3,2014-01-01,Antioquia,Angostura,Lesiones personales,CONTUNDENTES,Masculino,Adultos,1
4,2014-01-01,Antioquia,Angostura,Lesiones personales,VEHICULO,Femenino,Adultos,1


In [45]:
duplicados = df.duplicated(subset=columnas_dimension).sum()

print(f"Duplicados después de la consolidación: {duplicados:,}")

Duplicados después de la consolidación: 0


Almacenar los datos finales en un CSV

In [46]:
# Guardar el DataFrame transformado
ruta_salida = "Salidas/delitos_transformados.csv"

df.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo guardado en: {ruta_salida}")
print(f"Registros: {len(df):,}")

Archivo guardado en: Salidas/delitos_transformados.csv
Registros: 2,786,613
